# Final Model Comparison

This notebook compares four saved model runs using only CSV/JSON artifacts already written under `artifacts/`:

| Tag | Source notebook | Artifact root | Tensor bundle | Notes |
| --- | --- | --- | --- | --- |
| `mlp` | `4_Model_implementatin.ipynb` | `artifacts/lightning/linear_regression/drug_blind_mlp_delta/version_2` | `L1000` | Branched MLP |
| `random_forest` | `6_random_forest_model.ipynb` | `artifacts/sklearn/random_forest/drug_blind_random_forest_delta` | `L1000` | sklearn multi-output random forest |
| `adae_plate` | `8_adae_plate.ipynb` | `artifacts/lightning/adae_plate/drug_blind_adae_plate_delta/version_5` | `L1000_plate` | Plate-adversary AD-AE |
| `adae_plate_ablation_lambda0` | `8_adae_plate.ipynb` | `artifacts/lightning/adae_plate/drug_blind_adae_plate_delta_ablation_lambda0/version_3` | `L1000_plate` | Same architecture with lambda pinned to 0 |

No checkpoints are loaded and no inference is rerun here. The notebook validates the saved files, normalizes their naming differences, compares the exported metrics, and writes consolidated comparison tables back to `artifacts/comparisons/final_model_comparison`.

Because the `L1000` and `L1000_plate` bundles do not share identical `drug_blind` rows, cross-tensor comparisons should be interpreted with the printed `tensor_key` and `n_samples` columns in mind.


## Configuration


In [1]:
from pathlib import Path

SPLIT_MODE = "drug_blind"
RANDOM_SEED = 42

L1000_TENSOR_KEY = "L1000"
L1000_PLATE_TENSOR_KEY = "L1000_plate"
COMPARISON_OUTPUT_DIR = Path("artifacts/comparisons/final_model_comparison")

MODEL_TO_TENSOR_KEY = {
    "mlp": L1000_TENSOR_KEY,
    "random_forest": L1000_TENSOR_KEY,
    "adae_plate": L1000_PLATE_TENSOR_KEY,
    "adae_plate_ablation_lambda0": L1000_PLATE_TENSOR_KEY,
}
MODEL_DISPLAY_ORDER = [
    "mlp",
    "random_forest",
    "adae_plate",
    "adae_plate_ablation_lambda0",
]
MODEL_LABELS = {
    "mlp": "MLP",
    "random_forest": "Random Forest",
    "adae_plate": "ADAE Plate",
    "adae_plate_ablation_lambda0": "ADAE Plate lambda=0",
}
MODEL_COLOR_PALETTE = {
    "mlp": "#1f77b4",
    "random_forest": "#2ca02c",
    "adae_plate": "#d62728",
    "adae_plate_ablation_lambda0": "#ff7f0e",
}

MODEL_ARTIFACTS = {
    "mlp": {
        "artifact_root": Path("artifacts/lightning/linear_regression/drug_blind_mlp_delta/version_2"),
        "tables": {
            "evaluation_summary": ("evaluation_summary.csv", "evaluation_summary.json", True),
            "retrieval_summary": ("retrieval_summary.csv", "retrieval_summary.json", False),
            "test_prediction_details": ("test_prediction_details.csv", "test_prediction_details.json", True),
            "test_cell_line_accuracy": ("test_cell_line_accuracy.csv", "test_cell_line_accuracy.json", False),
            "test_cell_line_deg_match": ("test_cell_line_deg_match.csv", "test_cell_line_deg_match.json", False),
            "test_cell_line_signed_ndcg": ("test_cell_line_signed_ndcg.csv", "test_cell_line_signed_ndcg.json", False),
        },
        "training_run_summary": "training_run_summary.json",
        "metrics_csv": None,
    },
    "random_forest": {
        "artifact_root": Path("artifacts/sklearn/random_forest/drug_blind_random_forest_delta"),
        "tables": {
            "evaluation_summary": ("evaluation_summary.csv", "evaluation_summary.json", True),
            "retrieval_summary": ("retrieval_summary.csv", "retrieval_summary.json", False),
            "test_prediction_details": ("test_prediction_details.csv", "test_prediction_details.json", True),
            "test_cell_line_accuracy": ("test_cell_line_accuracy.csv", "test_cell_line_accuracy.json", False),
            "test_cell_line_deg_match": ("test_cell_line_deg_match.csv", "test_cell_line_deg_match.json", False),
            "test_cell_line_signed_ndcg": ("test_cell_line_signed_ndcg.csv", "test_cell_line_signed_ndcg.json", False),
        },
        "training_run_summary": "training_run_summary.json",
        "metrics_csv": None,
    },
    "adae_plate": {
        "artifact_root": Path("artifacts/lightning/adae_plate/drug_blind_adae_plate_delta/version_5"),
        "tables": {
            "evaluation_summary": ("evaluation_summary.csv", "evaluation_summary.json", True),
            "retrieval_summary": ("retrieval_summary.csv", "retrieval_summary.json", False),
            "test_prediction_details": ("test_prediction_details.csv", "test_prediction_details.json", True),
            "plate_variance": ("main_plate_variance.csv", "main_plate_variance.json", False),
            "post_training_probe": ("post_training_probe.csv", "post_training_probe.json", False),
            "summary_comparison": ("summary_comparison.csv", "summary_comparison.json", False),
        },
        "training_run_summary": "training_run_summary.json",
        "metrics_csv": "metrics.csv",
    },
    "adae_plate_ablation_lambda0": {
        "artifact_root": Path(
            "artifacts/lightning/adae_plate/drug_blind_adae_plate_delta_ablation_lambda0/version_3"
        ),
        "tables": {
            "evaluation_summary": ("ablation_evaluation_summary.csv", "ablation_evaluation_summary.json", True),
            "test_prediction_details": ("ablation_test_prediction_details.csv", "ablation_test_prediction_details.json", True),
            "plate_variance": ("ablation_plate_variance.csv", "ablation_plate_variance.json", False),
            "post_training_probe": ("ablation_post_training_probe.csv", "ablation_post_training_probe.json", False),
        },
        "training_run_summary": "training_run_summary.json",
        "metrics_csv": "metrics.csv",
    },
}

MODEL_METRICS_CSV_PATHS = {
    model_name: spec["artifact_root"] / spec["metrics_csv"]
    for model_name, spec in MODEL_ARTIFACTS.items()
    if spec["metrics_csv"] is not None
}

MODEL_ARTIFACTS


{'mlp': PosixPath('artifacts/lightning/linear_regression/drug_blind_mlp_delta/checkpoints/epoch=epoch=00-val_loss=val_loss=591.767273.ckpt'),
 'adae_cell_line': PosixPath('artifacts/lightning/adae/drug_blind_adae_delta/checkpoints/epoch=epoch=02-val_treated_cosine=val_treated_cosine=0.981929.ckpt'),
 'adae_plate': PosixPath('artifacts/lightning/adae_plate/drug_blind_adae_plate_delta/checkpoints/epoch=epoch=01-val_delta_pearson=val_delta_pearson=0.432664.ckpt'),
 'adae_plate_ablation_lambda0': PosixPath('artifacts/lightning/adae_plate/drug_blind_adae_plate_delta_ablation_lambda0/checkpoints/epoch=epoch=02-val_delta_pearson=val_delta_pearson=0.400924.ckpt')}

## Imports and Project Root


In [2]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

for candidate_root in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate_root / "pyproject.toml").exists() and (candidate_root / "Machine_Learning" / "ml_pipeline").exists():
        candidate_root_str = str(candidate_root)
        if candidate_root_str not in sys.path:
            sys.path.insert(0, candidate_root_str)
        PROJECT_ROOT = candidate_root
        break
else:
    raise ModuleNotFoundError(
        "Could not resolve the project root needed to import Machine_Learning.ml_pipeline."
    )

from Machine_Learning.ml_pipeline.utils import (
    load_lightning_metrics_table,
    resolve_project_path,
    write_performance_metrics,
)

sns.set_theme(style="whitegrid")
PROJECT_ROOT


Seed set to 42


42

## Validate and Load Artifact Tables

Each model is described by a small manifest that points to the exact CSV/JSON files already saved in `artifacts/`. We validate required files up front, normalize the ablation naming convention, and keep the loaded tables in memory for the downstream comparison sections.


In [3]:
def _load_json(path):
    return json.loads(Path(path).read_text())


loaded_tables = {}
training_run_summaries = {}
artifact_manifest_rows = []
missing_required_paths = []

for model_name in MODEL_DISPLAY_ORDER:
    spec = MODEL_ARTIFACTS[model_name]
    artifact_root = resolve_project_path(spec["artifact_root"])
    model_tables = {}

    training_run_summary_path = artifact_root / spec["training_run_summary"]
    if not training_run_summary_path.exists():
        missing_required_paths.append(str(training_run_summary_path))
    else:
        training_run_summaries[model_name] = _load_json(training_run_summary_path)

    for table_name, (csv_name, json_name, required) in spec["tables"].items():
        csv_path = artifact_root / csv_name
        json_path = artifact_root / json_name
        csv_exists = csv_path.exists()
        json_exists = json_path.exists()

        if required and not csv_exists:
            missing_required_paths.append(str(csv_path))
        if required and not json_exists:
            missing_required_paths.append(str(json_path))
        if csv_exists and json_exists:
            model_tables[table_name] = pd.read_csv(csv_path)

        artifact_manifest_rows.append(
            {
                "model_name": model_name,
                "model_label": MODEL_LABELS[model_name],
                "tensor_key": MODEL_TO_TENSOR_KEY[model_name],
                "artifact_root": str(artifact_root),
                "table_name": table_name,
                "csv_path": str(csv_path),
                "json_path": str(json_path),
                "required": required,
                "csv_exists": csv_exists,
                "json_exists": json_exists,
            }
        )

    metrics_csv_name = spec["metrics_csv"]
    metrics_csv_path = artifact_root / metrics_csv_name if metrics_csv_name is not None else None
    metrics_csv_exists = metrics_csv_path is not None and metrics_csv_path.exists()
    if metrics_csv_path is not None and not metrics_csv_exists:
        missing_required_paths.append(str(metrics_csv_path))

    artifact_manifest_rows.append(
        {
            "model_name": model_name,
            "model_label": MODEL_LABELS[model_name],
            "tensor_key": MODEL_TO_TENSOR_KEY[model_name],
            "artifact_root": str(artifact_root),
            "table_name": "metrics_csv",
            "csv_path": str(metrics_csv_path) if metrics_csv_path is not None else "",
            "json_path": "",
            "required": metrics_csv_path is not None,
            "csv_exists": metrics_csv_exists,
            "json_exists": pd.NA,
        }
    )

    loaded_tables[model_name] = model_tables

if missing_required_paths:
    missing_list = "\n".join(f"- {path}" for path in sorted(set(missing_required_paths)))
    raise FileNotFoundError(f"Missing required comparison artifacts:\n{missing_list}")

artifact_manifest_df = pd.DataFrame(artifact_manifest_rows)
display(artifact_manifest_df)


,tensor_key,tensor_artifacts_dir,n_examples,n_train,n_val,n_test,n_unique_drugs,n_unique_cell_lines,n_unique_plates,target_gene_dim
0,L1000,/Users/liamwilson/dl_fin/deep-learning-final-p...,26696,21368,2664,2664,377,24,NaN,977
1,L1000_plate,/Users/liamwilson/dl_fin/deep-learning-final-p...,28304,22636,2830,2838,377,24,13.0,977


## Build Common Comparison Frames

After the raw tables are loaded, we normalize them into a few shared DataFrames: model-only evaluation rows, null-baseline rows, retrieval summaries, per-row test prediction details, plate-specific summaries, and model notes describing any artifact gaps.


In [4]:
def _normalize_evaluation_table(model_name, table_df):
    normalized = table_df.copy()
    if "predictor" not in normalized.columns:
        normalized["predictor"] = "model"
    normalized["predictor"] = normalized["predictor"].fillna("model")
    if "cell_line_mean_fell_back" not in normalized.columns:
        normalized["cell_line_mean_fell_back"] = pd.NA
    normalized["model_name"] = model_name
    normalized["model_label"] = MODEL_LABELS[model_name]
    normalized["tensor_key"] = MODEL_TO_TENSOR_KEY[model_name]
    return normalized


evaluation_with_baselines_df = pd.concat(
    [
        _normalize_evaluation_table(model_name, loaded_tables[model_name]["evaluation_summary"])
        for model_name in MODEL_DISPLAY_ORDER
    ],
    ignore_index=True,
)
evaluation_column_order = ["model_name", "model_label", "tensor_key", "predictor", "split", "n_samples"]
evaluation_with_baselines_df = evaluation_with_baselines_df.loc[
    :,
    evaluation_column_order
    + [
        column
        for column in evaluation_with_baselines_df.columns
        if column not in set(evaluation_column_order)
    ],
]

evaluation_summary_df = evaluation_with_baselines_df.loc[
    evaluation_with_baselines_df["predictor"] == "model"
].copy()
null_baseline_df = (
    evaluation_with_baselines_df.loc[evaluation_with_baselines_df["predictor"] != "model"]
    .sort_values(["tensor_key", "predictor", "split", "model_name"], ignore_index=True)
    .drop_duplicates(subset=["tensor_key", "predictor", "split"], keep="first")
    .copy()
)

prediction_details_by_model = {}
all_test_prediction_details = []
for model_name in MODEL_DISPLAY_ORDER:
    details_df = loaded_tables[model_name]["test_prediction_details"].copy()
    details_df["model_name"] = model_name
    details_df["model_label"] = MODEL_LABELS[model_name]
    details_df["tensor_key"] = MODEL_TO_TENSOR_KEY[model_name]
    prediction_details_by_model[model_name] = {"test": details_df}
    all_test_prediction_details.append(details_df)
all_test_prediction_details_df = pd.concat(all_test_prediction_details, ignore_index=True)

retrieval_frames = []
retrieval_availability_rows = []
for model_name in MODEL_DISPLAY_ORDER:
    retrieval_df = loaded_tables[model_name].get("retrieval_summary")
    retrieval_available = retrieval_df is not None and not retrieval_df.empty
    retrieval_availability_rows.append(
        {
            "model_name": model_name,
            "model_label": MODEL_LABELS[model_name],
            "tensor_key": MODEL_TO_TENSOR_KEY[model_name],
            "retrieval_available": retrieval_available,
        }
    )
    if retrieval_available:
        retrieval_df = retrieval_df.copy()
        retrieval_df["model_name"] = model_name
        retrieval_df["model_label"] = MODEL_LABELS[model_name]
        retrieval_df["tensor_key"] = MODEL_TO_TENSOR_KEY[model_name]
        retrieval_frames.append(retrieval_df)
retrieval_availability_df = pd.DataFrame(retrieval_availability_rows)
retrieval_summary_df = pd.concat(retrieval_frames, ignore_index=True) if retrieval_frames else pd.DataFrame()
retrieval_comparison_df = (
    retrieval_summary_df.loc[retrieval_summary_df["split"] == "test"].copy()
    if not retrieval_summary_df.empty
    else pd.DataFrame()
)

plate_variance_frames = []
probe_frames = []
for model_name in ("adae_plate", "adae_plate_ablation_lambda0"):
    plate_variance_table = loaded_tables[model_name].get("plate_variance")
    if plate_variance_table is not None and not plate_variance_table.empty:
        plate_variance_table = plate_variance_table.copy()
        plate_variance_table["model_name"] = model_name
        plate_variance_table["model_label"] = MODEL_LABELS[model_name]
        plate_variance_frames.append(plate_variance_table)

    probe_table = loaded_tables[model_name].get("post_training_probe")
    if probe_table is not None and not probe_table.empty:
        probe_table = probe_table.copy()
        probe_table["model_name"] = model_name
        probe_table["model_label"] = MODEL_LABELS[model_name]
        probe_frames.append(probe_table)

plate_variance_df = pd.concat(plate_variance_frames, ignore_index=True) if plate_variance_frames else pd.DataFrame()
probe_df = pd.concat(probe_frames, ignore_index=True) if probe_frames else pd.DataFrame()
summary_comparison_df = loaded_tables["adae_plate"].get("summary_comparison", pd.DataFrame()).copy()
plate_specific_comparison_df = summary_comparison_df.copy()

model_notes_rows = []
for model_name in MODEL_DISPLAY_ORDER:
    table_names = sorted(loaded_tables[model_name].keys())
    notes = []
    if model_name == "adae_plate_ablation_lambda0":
        notes.append("No retrieval summary was exported for the ablation run.")
    if MODEL_TO_TENSOR_KEY[model_name] == L1000_PLATE_TENSOR_KEY:
        notes.append("Uses the L1000_plate split, which is not row-identical to L1000.")
    else:
        notes.append("Uses the L1000 split.")
    if model_name in MODEL_METRICS_CSV_PATHS:
        notes.append("Epoch-level Lightning metrics.csv is available.")
    else:
        notes.append("No epoch-level Lightning metrics.csv is available.")

    model_notes_rows.append(
        {
            "model_name": model_name,
            "model_label": MODEL_LABELS[model_name],
            "tensor_key": MODEL_TO_TENSOR_KEY[model_name],
            "artifact_root": str(resolve_project_path(MODEL_ARTIFACTS[model_name]["artifact_root"])),
            "available_tables": ", ".join(table_names),
            "has_retrieval_summary": bool(
                retrieval_availability_df.loc[
                    retrieval_availability_df["model_name"] == model_name,
                    "retrieval_available",
                ].iloc[0]
            ),
            "has_metrics_csv": model_name in MODEL_METRICS_CSV_PATHS,
            "notes": " ".join(notes),
        }
    )
model_notes_df = pd.DataFrame(model_notes_rows)


RuntimeError: Error(s) in loading state_dict for ADAEDrugResponseModule:
	size mismatch for adversary_head.0.weight: copying a param with shape torch.Size([100, 64]) from checkpoint, the shape in current model is torch.Size([100, 256]).

## Combined Evaluation Summary

These are the saved model rows loaded from each run's exported `evaluation_summary` table. The downstream notebook sections use these normalized rows rather than recomputing metrics from checkpoints.


In [ ]:
display(evaluation_summary_df)


## Null Baselines Already Exported By The Source Runs

Some source notebooks wrote `zero`, `global_mean`, and `cell_line_mean` rows into their saved `evaluation_summary` exports. We keep those rows separate here so the headline plots can still show a baseline reference without recomputation.


In [ ]:
display(
    null_baseline_df.sort_values(
        ["tensor_key", "split", "predictor", "model_name"],
        ignore_index=True,
    )
)


## Retrieval Metric Comparison

This section reads the saved `retrieval_summary` tables directly from `artifacts/`. The ablation run does not currently have a retrieval export, so it is listed as unavailable instead of being recomputed.


In [ ]:
if retrieval_comparison_df.empty:
    print("No retrieval_summary artifacts were found.")
else:
    display(
        retrieval_comparison_df.sort_values(
            ["tensor_key", "model_name"],
            ignore_index=True,
        )
    )

missing_retrieval_models = retrieval_availability_df.loc[
    ~retrieval_availability_df["retrieval_available"],
    "model_name",
].tolist()
if missing_retrieval_models:
    print(
        "Retrieval summary not available for: "
        + ", ".join(missing_retrieval_models)
        + ". Those rows are omitted rather than recomputed."
    )


## Head-to-Head Test Metric Table

A single table, one row per model, containing the headline test-set metrics. Use this as
the primary go/no-go comparison; per-row deltas from the best row are appended for each
metric where larger is better (so positive = worse than best).


In [ ]:
TEST_HEADLINE_METRICS = [
    ("delta_pearson_mean", True),
    ("delta_spearman_mean", True),
    ("delta_cosine_mean", True),
    ("treated_cosine", True),
    ("top50_deg_match_count_mean", True),
    ("signed_ndcg_at_50_mean", True),
    ("delta_mse", False),
    ("delta_mae", False),
    ("mann_whitney_not_significant_fraction", True),
]

test_summary_df = evaluation_summary_df.loc[evaluation_summary_df["split"] == "test"].copy()
test_summary_df["model_order"] = test_summary_df["model_name"].map(
    {name: idx for idx, name in enumerate(MODEL_DISPLAY_ORDER)}
)
test_summary_df = test_summary_df.sort_values("model_order", ignore_index=True).drop(columns=["model_order"])

test_headline_df = test_summary_df.loc[
    :,
    ["model_name", "model_label", "tensor_key", "n_samples"]
    + [name for name, _ in TEST_HEADLINE_METRICS],
].copy()
for metric_name, higher_is_better in TEST_HEADLINE_METRICS:
    gap_column = f"{metric_name}_gap_to_best"
    if higher_is_better:
        best_value = test_headline_df[metric_name].max()
        test_headline_df[gap_column] = best_value - test_headline_df[metric_name]
    else:
        best_value = test_headline_df[metric_name].min()
        test_headline_df[gap_column] = test_headline_df[metric_name] - best_value

display(test_headline_df)


## Headline Metric Bar Plots

The four most load-bearing metrics (delta Pearson, signed nDCG@50, top-50 DEG overlap,
treated cosine) on the test split, one model per bar. Dashed reference lines show the
strongest null baseline from `null_baseline_df` for the matching tensor set, so it is
immediately obvious whether each model beats "predict mean Δ per cell line".


In [ ]:
HEADLINE_BAR_METRICS = [
    ("delta_pearson_mean", "Test Mean Δ Pearson"),
    ("signed_ndcg_at_50_mean", "Test Mean Signed nDCG@50"),
    ("top50_deg_match_count_mean", "Test Mean Top-50 DEG Overlap Count"),
    ("treated_cosine", "Test Treated-Expression Cosine"),
]

MODEL_COLOR_PALETTE = {
    "mlp": "#1f77b4",
    "random_forest": "#2ca02c",
    "adae_cell_line": "#9467bd",
    "adae_plate": "#d62728",
    "adae_plate_ablation_lambda0": "#ff7f0e",
}

null_baseline_test_df = null_baseline_df.loc[
    (null_baseline_df["split"] == "test") & (null_baseline_df["predictor"] == "cell_line_mean")
].set_index("tensor_key")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
plot_order = [name for name in MODEL_DISPLAY_ORDER if name in set(test_summary_df["model_name"])]
for ax, (metric_name, metric_title) in zip(axes, HEADLINE_BAR_METRICS):
    sns.barplot(
        data=test_summary_df,
        x="model_name",
        y=metric_name,
        order=plot_order,
        hue="model_name",
        palette=MODEL_COLOR_PALETTE,
        dodge=False,
        legend=False,
        ax=ax,
    )
    for tensor_key, baseline_row in null_baseline_test_df.iterrows():
        baseline_value = float(baseline_row[metric_name]) if metric_name in baseline_row else float("nan")
        if not np.isnan(baseline_value):
            ax.axhline(
                baseline_value,
                linestyle="--",
                linewidth=1.0,
                color="#555555" if tensor_key == L1000_TENSOR_KEY else "#aa5555",
                label=f"cell_line_mean ({tensor_key})",
            )
    ax.set_title(metric_title)
    ax.set_xlabel("")
    ax.set_ylabel(metric_name)
    ax.tick_params(axis="x", rotation=30)
    ax.grid(True, axis="y", alpha=0.25)
    ax.legend(frameon=False, loc="best", fontsize=8)
fig.suptitle(f"Test-Set Model Comparison ({SPLIT_MODE}, seed {RANDOM_SEED})", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


## Training Dynamics Overlay

Only the two plate-model runs currently have saved `metrics.csv` files in `artifacts/`, so this overlay is limited to `adae_plate` and `adae_plate_ablation_lambda0`. The MLP and random forest runs still participate in the final comparison tables, but they do not contribute epoch-level traces here.


In [ ]:
def _epoch_collapsed_series(metrics_df, column_name):
    if column_name not in metrics_df.columns:
        return None
    series_df = metrics_df.loc[metrics_df[column_name].notna(), ["epoch", column_name]].copy()
    if series_df.empty:
        return None
    series_df = series_df.sort_values("epoch")
    return series_df.groupby("epoch", as_index=False)[column_name].last()


training_curves_rows = []
for model_name in MODEL_DISPLAY_ORDER:
    if model_name not in MODEL_METRICS_CSV_PATHS:
        continue
    csv_path = resolve_project_path(MODEL_METRICS_CSV_PATHS[model_name])
    metrics_df = load_lightning_metrics_table(csv_path)
    for metric_column in ("val_loss", "val_delta_pearson", "val_treated_cosine", "train_loss"):
        collapsed = _epoch_collapsed_series(metrics_df, metric_column)
        if collapsed is None:
            continue
        for _, curve_row in collapsed.iterrows():
            training_curves_rows.append(
                {
                    "model_name": model_name,
                    "metric": metric_column,
                    "epoch": int(curve_row["epoch"]),
                    "value": float(curve_row[metric_column]),
                }
            )
training_curves_df = pd.DataFrame(training_curves_rows)

available_metrics = sorted(training_curves_df["metric"].unique().tolist())
preferred_panel_metrics = [
    metric for metric in ("val_delta_pearson", "val_loss", "val_treated_cosine", "train_loss")
    if metric in available_metrics
]
if not preferred_panel_metrics:
    raise ValueError("No recognizable validation / training metric columns were logged for any model.")

fig, axes = plt.subplots(1, len(preferred_panel_metrics), figsize=(6 * len(preferred_panel_metrics), 5), sharex=False)
if len(preferred_panel_metrics) == 1:
    axes = [axes]
for ax, metric_name in zip(axes, preferred_panel_metrics):
    metric_df = training_curves_df.loc[training_curves_df["metric"] == metric_name]
    sns.lineplot(
        data=metric_df,
        x="epoch",
        y="value",
        hue="model_name",
        hue_order=[name for name in MODEL_DISPLAY_ORDER if name in set(metric_df["model_name"])],
        palette=MODEL_COLOR_PALETTE,
        marker="o",
        ax=ax,
    )
    ax.set_title(f"{metric_name} by Epoch")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(metric_name)
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False, fontsize=8, loc="best")
fig.suptitle(f"Training Dynamics ({SPLIT_MODE})", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


## Cell-Line Accuracy Under Mann-Whitney Non-Significance

Each prediction counts as "correct" if its per-row Mann-Whitney p-value is above 0.05 (i.e. the
predicted and true treated expression distributions are statistically indistinguishable). We
report per-cell-line accuracy across the test split for every model on the same axis so outliers
are obvious.


In [ ]:
cell_line_accuracy_rows = []
for model_name in MODEL_DISPLAY_ORDER:
    details = prediction_details_by_model[model_name].get("test")
    if details is None or details.empty:
        continue
    cell_line_details = details.copy()
    cell_line_details["is_correct_prediction"] = cell_line_details["mann_whitney_pvalue"] > 0.05
    per_cell_line_accuracy = (
        cell_line_details.groupby("cell_line", as_index=False)
        .agg(
            n_test_samples=("condition_key", "size"),
            n_correct_predictions=("is_correct_prediction", "sum"),
        )
    )
    per_cell_line_accuracy["n_correct_predictions"] = per_cell_line_accuracy["n_correct_predictions"].astype(int)
    per_cell_line_accuracy["cell_line_accuracy"] = (
        per_cell_line_accuracy["n_correct_predictions"] / per_cell_line_accuracy["n_test_samples"]
    )
    per_cell_line_accuracy["model_name"] = model_name
    cell_line_accuracy_rows.append(per_cell_line_accuracy)
cell_line_accuracy_df = pd.concat(cell_line_accuracy_rows, ignore_index=True) if cell_line_accuracy_rows else pd.DataFrame()

mean_cell_line_accuracy_df = (
    cell_line_accuracy_df.groupby("model_name", as_index=False)["cell_line_accuracy"].mean()
    if not cell_line_accuracy_df.empty
    else pd.DataFrame()
)
display(mean_cell_line_accuracy_df)

if not cell_line_accuracy_df.empty:
    cell_line_order = (
        cell_line_accuracy_df.groupby("cell_line", as_index=False)["cell_line_accuracy"]
        .mean()
        .sort_values("cell_line_accuracy", ascending=False)["cell_line"]
        .tolist()
    )
    fig, ax = plt.subplots(figsize=(16, 6))
    sns.barplot(
        data=cell_line_accuracy_df,
        x="cell_line",
        y="cell_line_accuracy",
        hue="model_name",
        hue_order=[name for name in MODEL_DISPLAY_ORDER if name in set(cell_line_accuracy_df["model_name"])],
        palette=MODEL_COLOR_PALETTE,
        order=cell_line_order,
        ax=ax,
    )
    ax.set_title("Test Cell-Line Accuracy (Mann-Whitney p > 0.05) by Model")
    ax.set_xlabel("Cell line")
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0, 1)
    ax.grid(True, axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=45)
    ax.legend(frameon=False, fontsize=8, loc="best")
    fig.tight_layout()
    plt.show()


## Per-Plate Variance: Plate Adversary vs lambda=0 Ablation

This section uses the saved plate-specific exports from notebook 8 rather than regrouping predictions on the fly. The main run provides `main_plate_variance` and `summary_comparison`, while the ablation run provides `ablation_plate_variance`; we normalize those into one side-by-side comparison.


In [ ]:
if plate_variance_df.empty:
    raise ValueError("Expected saved plate variance artifacts for the two plate-model runs.")

plate_variance_display_df = plate_variance_df.loc[
    :,
    [
        "model_name",
        "model_label",
        "n_plates",
        "treated_cosine_mean_of_plate_means",
        "treated_cosine_var_of_plate_means",
        "delta_mse_mean_of_plate_means",
        "delta_mse_var_of_plate_means",
        "delta_mse_range_of_plate_means",
    ],
].copy()
display(plate_variance_display_df)

if plate_specific_comparison_df.empty:
    plate_specific_comparison_df = plate_variance_display_df.copy()
else:
    display(plate_specific_comparison_df)

comparison_plot_df = summary_comparison_df.loc[
    summary_comparison_df["metric"].isin(
        [
            "treated_cosine_var_of_plate_means",
            "delta_mse_var_of_plate_means",
            "test_treated_cosine_mean",
            "test_delta_mse_mean",
        ]
    )
].copy()

if not comparison_plot_df.empty:
    comparison_plot_df["metric"] = pd.Categorical(
        comparison_plot_df["metric"],
        categories=[
            "treated_cosine_var_of_plate_means",
            "delta_mse_var_of_plate_means",
            "test_treated_cosine_mean",
            "test_delta_mse_mean",
        ],
        ordered=True,
    )
    comparison_plot_df = comparison_plot_df.sort_values("metric")
    melted_plot_df = comparison_plot_df.melt(
        id_vars="metric",
        value_vars=["main_adversary_on", "ablation_lambda0"],
        var_name="run",
        value_name="value",
    )
    melted_plot_df["run"] = melted_plot_df["run"].map(
        {
            "main_adversary_on": MODEL_LABELS["adae_plate"],
            "ablation_lambda0": MODEL_LABELS["adae_plate_ablation_lambda0"],
        }
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    variance_plot_df = melted_plot_df.loc[
        melted_plot_df["metric"].isin(
            ["treated_cosine_var_of_plate_means", "delta_mse_var_of_plate_means"]
        )
    ]
    performance_plot_df = melted_plot_df.loc[
        melted_plot_df["metric"].isin(["test_treated_cosine_mean", "test_delta_mse_mean"])
    ]

    palette = [
        MODEL_COLOR_PALETTE["adae_plate"],
        MODEL_COLOR_PALETTE["adae_plate_ablation_lambda0"],
    ]

    sns.barplot(
        data=variance_plot_df,
        x="metric",
        y="value",
        hue="run",
        palette=palette,
        ax=axes[0],
    )
    axes[0].set_title("Per-Plate Variance Metrics")
    axes[0].set_xlabel("")
    axes[0].tick_params(axis="x", rotation=25)
    axes[0].grid(True, axis="y", alpha=0.25)

    sns.barplot(
        data=performance_plot_df,
        x="metric",
        y="value",
        hue="run",
        palette=palette,
        ax=axes[1],
    )
    axes[1].set_title("Plate-Model Test Metrics")
    axes[1].set_xlabel("")
    axes[1].tick_params(axis="x", rotation=25)
    axes[1].grid(True, axis="y", alpha=0.25)

    for ax in axes:
        ax.legend(frameon=False, fontsize=8, loc="best")
    fig.tight_layout()
    plt.show()


## Linear Plate Probe on z_fused

Notebook 8 already exported post-training plate probe summaries for the main plate-adversary run and the lambda=0 ablation. We load those saved summaries directly here and compare them without rerunning the probe.


In [ ]:
if probe_df.empty:
    raise ValueError("Expected saved post-training probe artifacts for the two plate-model runs.")

probe_df = probe_df.loc[
    :,
    [
        "model_name",
        "model_label",
        "probe_accuracy",
        "random_baseline",
        "majority_baseline",
        "n_train_samples",
        "n_eval_samples",
        "n_train_classes",
        "n_eval_classes",
        "latent_dim",
    ],
].copy()
display(probe_df)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=probe_df,
    x="model_name",
    y="probe_accuracy",
    hue="model_name",
    palette=MODEL_COLOR_PALETTE,
    dodge=False,
    legend=False,
    ax=ax,
)
random_baseline = float(probe_df["random_baseline"].iloc[0])
majority_baseline = float(probe_df["majority_baseline"].max())
ax.axhline(
    random_baseline,
    linestyle="--",
    color="#555555",
    linewidth=1.0,
    label=f"random = {random_baseline:.3f}",
)
ax.axhline(
    majority_baseline,
    linestyle="--",
    color="#d62728",
    linewidth=1.0,
    label=f"majority = {majority_baseline:.3f}",
)
ax.set_ylim(0, max(1.0, float(probe_df["probe_accuracy"].max()) * 1.1))
ax.set_title("Linear Plate Probe Accuracy on z_fused (Test)")
ax.set_xlabel("")
ax.set_ylabel("Probe accuracy")
ax.grid(True, axis="y", alpha=0.25)
ax.legend(frameon=False, fontsize=8, loc="best")
fig.tight_layout()
plt.show()


## Exported Comparison Artifacts and Final Verdict

The last cell writes the consolidated comparison outputs to `artifacts/comparisons/final_model_comparison` and prints a compact verdict table for the four selected models.


In [ ]:
def _lookup_metric(test_df, model_name, column):
    row = test_df.loc[test_df["model_name"] == model_name]
    if row.empty or column not in row.columns:
        return float("nan")
    return float(row.iloc[0][column])


verdict_rows = []
for model_name in MODEL_DISPLAY_ORDER:
    verdict_rows.append(
        {
            "model_name": model_name,
            "model_label": MODEL_LABELS[model_name],
            "tensor_key": MODEL_TO_TENSOR_KEY[model_name],
            "test_n_samples": int(_lookup_metric(test_summary_df, model_name, "n_samples"))
            if model_name in set(test_summary_df["model_name"])
            else 0,
            "test_delta_pearson_mean": _lookup_metric(test_summary_df, model_name, "delta_pearson_mean"),
            "test_signed_ndcg_at_50_mean": _lookup_metric(test_summary_df, model_name, "signed_ndcg_at_50_mean"),
            "test_top50_deg_match_count_mean": _lookup_metric(test_summary_df, model_name, "top50_deg_match_count_mean"),
            "test_treated_cosine": _lookup_metric(test_summary_df, model_name, "treated_cosine"),
            "test_mann_whitney_not_sig_fraction": _lookup_metric(
                test_summary_df,
                model_name,
                "mann_whitney_not_significant_fraction",
            ),
        }
    )
verdict_df = pd.DataFrame(verdict_rows)

comparison_output_dir = PROJECT_ROOT / COMPARISON_OUTPUT_DIR
comparison_output_dir.mkdir(parents=True, exist_ok=True)

export_tables = {
    "test_headline_metrics": test_headline_df,
    "retrieval_comparison": retrieval_comparison_df,
    "plate_specific_comparison": plate_specific_comparison_df,
    "plate_probe_comparison": probe_df,
    "mean_cell_line_accuracy": mean_cell_line_accuracy_df,
    "verdict_summary": verdict_df,
    "model_notes": model_notes_df,
}
export_paths = write_performance_metrics(
    comparison_output_dir,
    export_tables,
    json_summaries={
        "artifact_manifest": artifact_manifest_df.to_dict(orient="records"),
        "training_run_summaries": training_run_summaries,
    },
)

display(verdict_df)
display(model_notes_df)
display(
    pd.DataFrame(
        [
            {"artifact_name": artifact_name, "path": str(path)}
            for artifact_name, path in sorted(export_paths.items())
        ]
    )
)

print(
    "Reminder: `adae_plate` and `adae_plate_ablation_lambda0` use the L1000_plate test split, "
    "while `mlp` and `random_forest` use the L1000 test split."
)
